In [4]:
!pip install fastapi uvicorn pyngrok python-multipart nest-asyncio
!pip install pikepdf
!pip install modelscope -q
!pip install transformers==4.49.0 -q
!pip install magic-pdf[full]==1.3.12 -q
!pip install ultralytics -q

In [5]:
# Cell 2: Download models

from modelscope import snapshot_download
import json, os

print('Đang tải PDF-Extract-Kit từ ModelScope...')
snapshot_download(
    'opendatalab/PDF-Extract-Kit-1.0',
    local_dir='/root/models/PDF-Extract-Kit-1.0'
)

print('Đang tải LayoutReader...')
snapshot_download(
    'ppaanngggg/layoutreader',
    local_dir='/root/models/layoutreader'
)

config = {
    'models-dir': '/root/models/PDF-Extract-Kit-1.0/models',
    'layoutreader-model-dir': '/root/models/layoutreader',
    'device-mode': 'cuda',
    'layout-config': {'model': 'doclayout_yolo'},
    'formula-config': {'mfd_model': 'yolo_v8_mfd', 'mfr_model': 'unimernet_small'},
    'table-config': {'model': 'rapid_table', 'is_table_recog_enable': False}
}
config_path = os.path.expanduser('~/magic-pdf.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print('✅ Xong')

In [6]:
# Cell 3: Verify GPU + Config
import subprocess, torch, json, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('KHÔNG CÓ GPU! Vào Settings → Accelerator → GPU T4 rồi restart.')
print(result.stdout)
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

config_path = os.path.expanduser('~/magic-pdf.json')
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print(f"\ndevice-mode : {config.get('device-mode')}")
    print(f"models-dir  : {config.get('models-dir')}")
    ok = (config.get('device-mode') == 'cuda' and
          config.get('formula-config', {}).get('mfr_model') == 'unimernet_small')
    print('\n✅ Config OK' if ok else '\n❌ Config sai - Chạy lại Cell 2')
else:
    print('❌ Không tìm thấy magic-pdf.json')

In [30]:
import subprocess
subprocess.run(["pip", "install", "transformers==4.49.0", "-q"],
               capture_output=True)
print("✅ transformers 4.49.0")
# Tự động tạo symlink OCR mỗi lần chạy Cell 4
import os
ocr_dir = "/root/models/PDF-Extract-Kit-1.0/models/OCR/paddleocr_torch"
symlinks = {
    "ch_PP-OCRv3_det_infer.pth": "ch_PP-OCRv5_det_infer.pth",
    "ch_PP-OCRv3_rec_infer.pth": "ch_PP-OCRv4_rec_infer.pth",
}
for link_name, target_name in symlinks.items():
    link_path = os.path.join(ocr_dir, link_name)
    target_path = os.path.join(ocr_dir, target_name)
    if os.path.islink(link_path) or os.path.exists(link_path):
        os.remove(link_path)
    if os.path.exists(target_path):
        os.symlink(target_path, link_path)
        print(f"✅ Symlink: {link_name}")

config_path = "/usr/local/lib/python3.12/dist-packages/magic_pdf/model/sub_modules/ocr/paddleocr2pytorch/pytorchocr/utils/resources/models_config.yml"
with open(config_path, "r") as f:
    content = f.read()
content = content.replace("ch_PP-OCRv3_det_infer.pth", "ch_PP-OCRv5_det_infer.pth")
with open(config_path, "w") as f:
    f.write(content)
print("✅ Patch models_config.yml")

# Cell 4: Server
import nest_asyncio, asyncio, re, json, time, os, shutil, glob, subprocess, fitz
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI, UploadFile, File, BackgroundTasks

PAGES_PER_PART = 10
NGROK_TOKEN = "3EOpbErom32LLC6yWu8LmgAdKeP_4qzLtnxN8P1vkW4Dff52p"
ngrok.set_auth_token(NGROK_TOKEN)
app = FastAPI()

# ============ HELPER ============
def sanitize_file_id(filename):
    name = filename.replace('.pdf', '')
    name = re.sub(r'[^\w\-]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name[:80]

def write_status(status_path, status, message=None):
    data = {'status': status, 'updated_at': time.time()}
    if message:
        data['message'] = message
    with open(status_path, 'w') as f:
        json.dump(data, f)

# ============ TÁCH PDF TRÊN COLAB ============
def split_pdf_on_colab(pdf_path, max_pages=PAGES_PER_PART):
    """
    Dùng pikepdf thay PyMuPDF để tách PDF.
    pikepdf tách theo cơ chế khác, không bị lỗi xref với MinerU.
    """
    import pikepdf

    pdf = pikepdf.open(pdf_path)
    total = len(pdf.pages)
    pdf.close()

    if total <= max_pages:
        print(f'  File {total} trang → không cần tách')
        return [pdf_path]

    print(f'  File {total} trang → tách thành các phần {max_pages} trang')
    parts = []
    base = pdf_path.replace('.pdf', '')

    for i, start in enumerate(range(0, total, max_pages)):
        end = min(start + max_pages, total)  # pikepdf dùng exclusive end
        part_path = f'{base}_part{i+1}.pdf'

        src = pikepdf.open(pdf_path)
        dst = pikepdf.Pdf.new()
        dst.pages.extend(src.pages[start:end])
        dst.save(part_path)
        src.close()
        dst.close()

        parts.append(part_path)
        print(f'  ✅ part{i+1}: trang {start+1}-{end}')

    return parts

# ============ MINERU TỪNG PART ============
def run_mineru_single(input_path, output_dir):
    """Chạy magic-pdf cho 1 file, trả về text hoặc None nếu lỗi."""
    log_path = os.path.join(output_dir, 'mineru.log')
    os.makedirs(output_dir, exist_ok=True)

    with open(log_path, 'w') as log_file:
        process = subprocess.Popen(
            ['magic-pdf', '-p', input_path, '-o', output_dir, '-m', 'ocr'],
            stdout=log_file, stderr=log_file
        )
        prev_size = 0
        while process.poll() is None:
            time.sleep(30)
            size = os.path.getsize(log_path) if os.path.exists(log_path) else 0
            s = '📈 đang xử lý' if size > prev_size else '⚠️ không thay đổi'
            print(f'    📄 log: {size} bytes — {s}')
            prev_size = size

    md_files = glob.glob(f'{output_dir}/**/*.md', recursive=True)
    md_files = [f for f in md_files if not f.endswith('layout.md')]

    if md_files:
        with open(md_files[0], 'r', encoding='utf-8') as f:
            return f.read()
    return None

# ============ MINERU TOÀN BỘ FILE ============
def run_mineru_with_split(input_path, output_dir):
    """Tách PDF trên Colab rồi chạy MinerU từng part, ghép kết quả."""
    status_path = os.path.join(output_dir, 'STATUS.json')
    write_status(status_path, 'running')

    try:
        parts = split_pdf_on_colab(input_path)
        all_texts = []

        for idx, part_path in enumerate(parts):
            part_name = os.path.basename(part_path)
            part_out  = part_path.replace('.pdf', '_out')
            print(f'\n▶ [{idx+1}/{len(parts)}] Xử lý: {part_name}')

            text = run_mineru_single(part_path, part_out)

            if text is None:
                write_status(status_path, 'error',
                    f'{part_name}: MinerU không sinh ra file .md (VRAM? PDF lỗi?)')
                print(f'  ❌ {part_name} thất bại')
                return

            all_texts.append(text)
            print(f'  ✅ {part_name} hoàn thành')

        # Ghi file md tổng
        merged_md = os.path.join(output_dir, 'merged.md')
        with open(merged_md, 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(all_texts))

        write_status(status_path, 'done')
        print(f'\n✅ Hoàn thành tất cả {len(parts)} parts')

    except Exception as e:
        write_status(status_path, 'error', str(e))
        print(f'  ❌ Exception: {e}')
        raise

    finally:
        try:
            with open(status_path) as f:
                cur = json.load(f)
            if cur.get('status') == 'running':
                write_status(status_path, 'error', 'Bị kill đột ngột (VRAM? OOM?)')
                print('  ❌ Process bị kill đột ngột')
        except:
            pass

# ============ API ============
@app.post('/convert')
async def convert_pdf(background_tasks: BackgroundTasks, file: UploadFile = File(...)):
    file_id    = sanitize_file_id(file.filename)
    input_path = f'input_{file_id}.pdf'
    output_dir = f'output_{file_id}'

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    with open(input_path, 'wb') as f:
        f.write(await file.read())

    background_tasks.add_task(run_mineru_with_split, input_path, output_dir)
    return {'status': 'processing', 'file_id': file_id}

@app.get('/result/{file_id}')
def get_result(file_id: str):
    output_dir  = f'output_{file_id}'
    status_path = os.path.join(output_dir, 'STATUS.json')

    if not os.path.exists(status_path):
        return {'status': 'processing'}

    with open(status_path) as f:
        status_data = json.load(f)
    status = status_data.get('status')

    if status == 'error':
        return {'status': 'error', 'message': status_data.get('message')}

    if status == 'running':
        return {'status': 'processing'}

    if status == 'done':
        merged_md = os.path.join(output_dir, 'merged.md')
        if os.path.exists(merged_md):
            with open(merged_md, 'r', encoding='utf-8') as f:
                return {'status': 'done', 'text': f.read()}
        return {'status': 'error', 'message': 'STATUS done nhưng không tìm thấy merged.md'}

    return {'status': 'processing'}

# ============ KHỞI ĐỘNG SERVER ============
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
        print(f'Đã đóng tunnel cũ: {tunnel.public_url}')
except:
    pass

ngrok.kill()
time.sleep(3)

try:
    public_url = ngrok.connect(8000).public_url
    print(f'\n🚀 LINK API MINERU TRÊN COLAB: {public_url}')
    print('Copy URL này vào biến COLAB_API_URL trong file .env của bạn\n')
except Exception as e:
    print(f'❌ Ngrok lỗi: {e}')
    print('Vào https://dashboard.ngrok.com/endpoints → xóa tunnel cũ → chạy lại cell')
    raise

nest_asyncio.apply()
config = uvicorn.Config(app, host='0.0.0.0', port=8000, loop='asyncio', log_level='info')
server = uvicorn.Server(config)

print('Server đang chạy. Giữ cell này chạy liên tục.')
print('Nếu cell dừng = server chết = cần chạy lại cell này.')

await server.serve()